# 🧮 LoRA 数学原理 — 低秩适配的完整推导

**论文**: [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) (Hu et al., ICLR 2022)

**本文目标**：从矩阵分解到梯度传播，完整理解 LoRA 为什么能用 0.1% 的参数微调大模型。

读完这篇你会理解：
- 什么是"低秩假设"——微调只是在一个低维子空间里移动
- SVD 与 LoRA 的关系
- 前向和反向传播的完整数学推导
- rank 和 alpha 的真正含义

## 1. 问题: 全量微调为什么不可行

```
全量微调 LLaMA-7B:
  训练参数: 7B × 4 bytes(FP32) = 28 GB  ← 仅模型权重
  优化器状态 (Adam): 28 × 3 = 84 GB        ← m, v, grad
  总计: 112 GB → 需要 4×A100(80GB)

  每次迭代的通信: 7B 个梯度 → ~28 GB all-reduce

LoRA:
  训练参数: ~10M (rank=16) × 4 bytes = 40 MB
  优化器状态: 40 × 3 = 120 MB
  总计: 160 MB → 1×RTX 4090(24GB) 绰绰有余

  参数量对比: 10M / 7B = 0.14%
```

## 2. 核心假设: 微调是"低秩"的

### 2.1 论文的关键发现

> "We take inspiration from Li et al. (2018a); Aghajanyan et al. (2020) which show that the learned over-parametrized models in fact reside on a low intrinsic dimension."

翻译：大模型虽然参数多，但**微调时只需要在一个低维子空间里调整**。

```python
# 全量微调: 更新整个权重矩阵
W_new = W_pretrained + ΔW
# ΔW ∈ R^(d×d), 如 4096×4096 = 16M 参数

# LoRA: 把 ΔW 分解为两个小矩阵的乘积
ΔW = B @ A
# A ∈ R^(r×d), B ∈ R^(d×r)
# 如 r=16: 16×4096 + 4096×16 = 131K 参数
# 压缩比: 16M / 131K ≈ 122x
```

### 2.2 为什么"低秩"假设成立？

```
SVD 视角:
  W_pretrained = U @ Σ @ V^T

  全量微调 ΔW 的奇异值谱:
    σ₁ ≥ σ₂ ≥ ... ≥ σ_d

  实验发现 (Aghajanyan et al., 2020):
    前 10% 的奇异值贡献了 > 90% 的能量
    → ΔW 可以被一个低秩矩阵很好地近似
    → rank(ΔW_effective) << d

  直观理解:
    微调时模型不需要"重新学习一切"
    只需要调整少数几个"方向" (如: 对特定领域的关注模式)
    → 这些方向对应 ΔW 的主要奇异向量
```

### 2.3 Forward Pass 推导

```python
# LoRA 的前向传播
def lora_forward(x, W_pretrained, A, B, alpha, dropout=0.0):
    """
    x: [batch, seq, d_in]
    W: [d_in, d_out] — 预训练权重 (frozen)
    A: [r, d_in]     — LoRA A (可训练)
    B: [d_out, r]    — LoRA B (可训练)
    alpha: 缩放因子
    """
    # 原始输出
    original = x @ W_pretrained.T  # [batch, seq, d_out]

    # LoRA 修正
    lora_out = (x @ A.T) @ B.T    # [batch, seq, r] @ [r, d_out]

    # 合并 (alpha/r 是缩放因子)
    output = original + (alpha / r) * lora_out
    return output

# 初始化策略:
# A: Kaiming uniform
# B: zeros  ← 关键! 保证初始时 LoRA 不改变原始输出
```

### 2.4 缩放因子的含义

```
output = W @ x + (alpha / r) × B @ A @ x

alpha / r 控制 LoRA 修正的强度:
  alpha = r:    缩放 = 1   (LoRA 修正不加权)
  alpha = 2r:   缩放 = 2   (LoRA 修正加倍)
  alpha = r/2:  缩放 = 0.5 (LoRA 修正减半)

经验法则:
  rank=8  → alpha=16  (缩放=2)
  rank=16 → alpha=16  (缩放=1, 最常用)
  rank=32 → alpha=32  (缩放=1)
  rank=64 → alpha=64  (缩放=1)

为什么需要 alpha?
  → 控制 LoRA 的学习率 vs 原始权重的比例
  → alpha 越大, LoRA 修正幅度越大 (但也可能破坏预训练知识)
```

### 2.5 Backward Pass

```python
# LoRA 的反向传播
# 只有 A 和 B 需要梯度, W 是 frozen

# 损失对输出的梯度
dL_doutput = ...  # [batch, seq, d_out]

# B 的梯度
dL_dB = (x @ A.T).T @ dL_doutput  # [d_out, r]

# A 的梯度
dL_dA = (dL_doutput @ B).T @ x    # [r, d_in]

# W 不需要更新 (frozen)
# → 反向传播时跳过 W 的梯度计算
# → 省 99.9% 的梯度计算和优化器状态!
```

## 3. LoRA 的变体选择: 哪些权重加 LoRA？

论文实验了不同的目标权重组合：

| 目标 | 参数量 (r=8) | 效果 (GPT-3 175B) |
|------|-------------|-------------------|
| 仅 Q | 4.7M | 基准 |
| 仅 K | 4.7M | 略差 |
| 仅 V | 4.7M | 接近 Q |
| **Q + V** | 9.4M | **最佳性价比** |
| Q + K + V + O | 18.8M | 略好, 但参数翻倍 |
| 仅 O | 4.7M | 差 |

> 论文推荐: **Q + V** (或 **Q + K + V + O** 如果不在乎参数量)

## 4. Rank 的选择

```python
# 不同 rank 的效果 (论文 Table 5, GPT-3 175B)

# rank=1:   效果已经不错 (说明微调确实极为低秩)
# rank=4:   接近 rank=8
# rank=8:   ≈ 饱和点
# rank=16:  与 rank=8 几乎无差异
# rank=64:  效果略好但参数 8x

# 经验:
# 通用微调: rank=8 or 16
# 领域适配: rank=16 or 32
# 复杂任务 (代码/数学): rank=32 or 64
# 位置编码扩展: rank=4 or 8 (极为低秩的适配)
```

In [ ]:
# LoRA 的核心概念验证

import torch
import torch.nn as nn

# 模拟一个简化的 attention 层
class SimpleAttention(nn.Module):
    def __init__(self, d_model=64):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        scores = q @ k.transpose(-2, -1)
        attn = torch.softmax(scores, dim=-1)
        return attn @ v

class LoRALinear(nn.Module):
    """LoRA 包装的线性层"""
    def __init__(self, linear, rank=4, alpha=8):
        super().__init__()
        self.linear = linear  # 原始权重 (frozen)
        d_in, d_out = linear.weight.shape
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        self.rank = rank
        self.alpha = alpha

    def forward(self, x):
        original = self.linear(x)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        return original + (self.alpha / self.rank) * lora_out

# 测试
base = SimpleAttention(d_model=64)
x = torch.randn(2, 10, 64)

# 原始输出
with torch.no_grad():
    out_base = base(x)

# 注入 LoRA
base.W_q = LoRALinear(base.W_q, rank=4, alpha=8)

# LoRA 初始化为 B=0 → 输出不变
out_lora_init = base(x)
print(f"LoRA 初始化验证:")
print(f"  原始输出 mean: {out_base.mean().item():.6f}")
print(f"  LoRA 输出 mean: {out_lora_init.mean().item():.6f}")
print(f"  差异: {(out_base - out_lora_init).abs().max().item():.10f}  ← 应为 0")

# 统计参数量
total = sum(p.numel() for p in base.parameters())
trainable = sum(p.numel() for p in base.parameters() if p.requires_grad)
print(f"\n参数量统计:")
print(f"  总参数: {total:,}")
print(f"  可训练 (LoRA): {trainable:,}")
print(f"  比例: {trainable/total*100:.2f}%")